In [1]:
import os, json, time, glob, warnings
import numpy as np, pandas as pd
warnings.filterwarnings("ignore")

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (accuracy_score, balanced_accuracy_score,
                             f1_score, matthews_corrcoef, confusion_matrix)
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

assert tf.config.list_physical_devices("GPU"), "NO GPU — stop and fix"

EPOCHS, BATCH_SIZE, TEST_SIZE = 40, 512, 0.30
LR, MOMENTUM = 0.01, 0.9
RESULTS_PATH = "/kaggle/working/rescued_perclass.json"

hits = glob.glob("/kaggle/input/**/ciciot2023_working_set.parquet", recursive=True)
df = pd.read_parquet(hits[0])
feature_cols = [c for c in df.columns if c != "family"]
X_all = df[feature_cols].to_numpy(dtype=np.float32)
le = LabelEncoder(); y_all = le.fit_transform(df["family"].to_numpy())
CLASS_NAMES = list(le.classes_)
N_FEATURES, N_CLASSES = X_all.shape[1], len(CLASS_NAMES)
print(f"X: {X_all.shape} | {N_CLASSES} classes: {CLASS_NAMES}")

def build_model(n_inputs, n_output, loss_fn):
    nb = int(round(n_inputs / 2.0))
    visible = keras.Input(shape=(n_inputs, 1))
    e = layers.Dense(n_inputs)(visible); e = layers.BatchNormalization()(e); e = layers.LeakyReLU()(e)
    bn = layers.Dense(nb)(e)
    d = layers.Dense(n_inputs)(bn); d = layers.BatchNormalization()(d); d = layers.LeakyReLU()(d)
    lstm = layers.LSTM(nb, activation="tanh", return_sequences=True)(visible)
    lstm = layers.Dense(n_inputs)(lstm)
    c = layers.Concatenate()([d, lstm])
    c = layers.Conv1D(filters=nb, kernel_size=2, activation="relu")(c)
    c = layers.Flatten()(c)
    out = layers.Dense(n_output, activation="softmax")(c)
    m = keras.Model(visible, out)
    m.compile(optimizer=keras.optimizers.SGD(learning_rate=LR, momentum=MOMENTUM),
              loss=loss_fn, metrics=["accuracy"])
    return m

def categorical_focal_loss(class_weights, gamma=2.0):
    w = tf.constant(class_weights, dtype=tf.float32)
    def loss(y_true, y_pred):
        y_pred = tf.clip_by_value(y_pred, 1e-7, 1.0 - 1e-7)
        ce = -y_true * tf.math.log(y_pred)
        return tf.reduce_sum(w * tf.pow(1.0 - y_pred, gamma) * ce, axis=-1)
    return loss

def run(strategy, seed, protocol="B"):
    t0 = time.time()
    np.random.seed(seed); tf.random.set_seed(seed)
    idx = np.arange(len(X_all))
    idx_tr, idx_te = train_test_split(idx, test_size=TEST_SIZE,
                                      random_state=seed, stratify=y_all)
    sc = StandardScaler().fit(X_all[idx_tr])
    X_tr, y_tr = sc.transform(X_all[idx_tr]), y_all[idx_tr]
    X_te, y_te = sc.transform(X_all[idx_te]), y_all[idx_te]

    i_fit, i_val = train_test_split(np.arange(len(y_tr)), test_size=0.10,
                                    random_state=seed, stratify=y_tr)
    X_fit, y_fit = X_tr[i_fit], y_tr[i_fit]
    X_val, y_val = X_tr[i_val], y_tr[i_val]

    if strategy == "focal":
        cnt = np.bincount(y_fit, minlength=N_CLASSES).astype(np.float64)
        cnt[cnt == 0] = 1.0
        cw = cnt.sum() / (N_CLASSES * cnt); cw = cw / cw.mean()
        loss_fn = categorical_focal_loss(cw.astype(np.float32))
    else:
        loss_fn = "categorical_crossentropy"

    rs = lambda a: a.reshape(-1, N_FEATURES, 1).astype(np.float32)
    X_fit, X_val, X_te_r = rs(X_fit), rs(X_val), rs(X_te)
    y_fit_oh = keras.utils.to_categorical(y_fit, N_CLASSES)
    y_val_oh = keras.utils.to_categorical(y_val, N_CLASSES)

    ckpt = f"/kaggle/working/_ck_{strategy}_{seed}.weights.h5"
    model = build_model(N_FEATURES, N_CLASSES, loss_fn)
    hist = model.fit(X_fit, y_fit_oh, epochs=EPOCHS, batch_size=BATCH_SIZE,
                     verbose=0, validation_data=(X_val, y_val_oh),
                     callbacks=[keras.callbacks.ModelCheckpoint(
                         ckpt, monitor="val_accuracy", mode="max",
                         save_best_only=True, save_weights_only=True)])

    def ev(m):
        p = m.predict(X_te_r, batch_size=2048, verbose=0).argmax(1)
        cm = confusion_matrix(y_te, p)
        return {"accuracy": float(accuracy_score(y_te, p)),
                "balanced_accuracy": float(balanced_accuracy_score(y_te, p)),
                "macro_f1": float(f1_score(y_te, p, average="macro", zero_division=0)),
                "mcc": float(matthews_corrcoef(y_te, p)),
                "confusion_matrix": cm.tolist()}

    fin = ev(model)
    model.load_weights(ckpt)
    res_ = ev(model)
    os.remove(ckpt)

    out = {"strategy": strategy, "protocol": protocol, "seed": seed,
           **{k: fin[k] for k in ["accuracy","balanced_accuracy","macro_f1","mcc"]},
           "confusion_matrix": fin["confusion_matrix"],
           "rescued_accuracy": res_["accuracy"],
           "rescued_balanced_accuracy": res_["balanced_accuracy"],
           "rescued_macro_f1": res_["macro_f1"],
           "rescued_mcc": res_["mcc"],
           "rescued_confusion_matrix": res_["confusion_matrix"],
           "best_val_epoch": int(np.argmax(hist.history["val_accuracy"])) + 1,
           "final_train_loss": float(hist.history["loss"][-1]),
           "wall_sec": round(time.time() - t0, 1)}
    keras.backend.clear_session()
    return out

print("Ready.")

X: (547944, 44) | 8 classes: ['Benign', 'BruteForce', 'DDoS', 'DoS', 'Mirai', 'Recon', 'Spoofing', 'Web']
Ready.


In [2]:
N_SEEDS = 10
results = json.load(open(RESULTS_PATH)) if os.path.exists(RESULTS_PATH) else []
done = {(r["strategy"], r["seed"]) for r in results}
grid = [(s, sd) for s in ["none", "focal"] for sd in range(N_SEEDS)]
print(f"{len(grid)} runs, {len(done)} done\n" + "="*72)

def web_recall(cm):
    cm = np.array(cm); i = CLASS_NAMES.index("Web")
    return cm[i, i] / max(cm[i].sum(), 1)

for k, (strat, seed) in enumerate(grid, 1):
    if (strat, seed) in done:
        print(f"[{k}/{len(grid)}] skip {strat}/s{seed}"); continue
    print(f"[{k}/{len(grid)}] {strat} seed={seed} ...", flush=True)
    r = run(strat, seed)
    results.append(r)
    json.dump(results, open(RESULTS_PATH, "w"))
    print(f"    final acc={r['accuracy']:.4f} Web={web_recall(r['confusion_matrix']):.4f}")
    print(f"    rescued  acc={r['rescued_accuracy']:.4f} Web={web_recall(r['rescued_confusion_matrix']):.4f}"
          f"  (ep {r['best_val_epoch']}, {r['wall_sec']}s)")

print("\nDone ->", RESULTS_PATH)

20 runs, 0 done
[1/20] none seed=0 ...


I0000 00:00:1786638028.886530      58 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1786638028.889567      58 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


    final acc=0.8643 Web=0.0162
    rescued  acc=0.8848 Web=0.0236  (ep 32, 236.9s)
[2/20] none seed=1 ...
    final acc=0.8020 Web=0.0292
    rescued  acc=0.8786 Web=0.0325  (ep 33, 235.1s)
[3/20] none seed=2 ...
    final acc=0.8119 Web=0.0276
    rescued  acc=0.8825 Web=0.0439  (ep 25, 232.3s)
[4/20] none seed=3 ...
    final acc=0.8860 Web=0.0390
    rescued  acc=0.8860 Web=0.0390  (ep 40, 232.0s)
[5/20] none seed=4 ...
    final acc=0.8745 Web=0.0366
    rescued  acc=0.8816 Web=0.0471  (ep 39, 232.1s)
[6/20] none seed=5 ...
    final acc=0.8362 Web=0.0374
    rescued  acc=0.8808 Web=0.0577  (ep 38, 231.7s)
[7/20] none seed=6 ...
    final acc=0.8134 Web=0.0406
    rescued  acc=0.8671 Web=0.0357  (ep 14, 231.0s)
[8/20] none seed=7 ...
    final acc=0.8260 Web=0.0593
    rescued  acc=0.8695 Web=0.0723  (ep 20, 232.8s)
[9/20] none seed=8 ...
    final acc=0.7505 Web=0.0723
    rescued  acc=0.8722 Web=0.0260  (ep 30, 232.5s)
[10/20] none seed=9 ...
    final acc=0.7374 Web=0.0536
    